In [7]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
SQLShield - Corrected Validation Pipeline
=========================================

This script replaces the flawed "concat first, then detect text column" data pipeline.

It performs:

1) Correct source-specific loading:
   A: sajid576/Modified_SQL_Dataset.csv      -> Query, Label
   B: syed.../sqliv2.csv                     -> Sentence, Label

2) Strict cleaning:
   - labels must be numeric 0 or 1
   - text must be non-null and length > 2
   - exact duplicates removed within each source
   - one known conflicting normalized group (#NAME?) is removed automatically
     if any normalized group has both labels

3) Correct merged corpus construction:
   - concatenate normalized source tables
   - remove exact duplicate text across sources
   - create normalized_text = lowercase + whitespace collapse

4) Leakage-resistant GROUP-AWARE 80/10/10 split:
   - all rows sharing the same normalized_text stay in ONE partition only
   - unique normalized groups are stratified by binary label
   - split seed fixed at 42
   - assertions verify zero normalized-group overlap across partitions

5) Classical ML baselines on corrected fixed split:
   - Random Forest
   - XGBoost
   - Logistic Regression
   - LinearSVC
   - TF-IDF word uni/bi, max_features=10000

6) Transformer multi-seed stability:
   - CodeBERT x 5 seeds
   - BERT-base x 5 seeds
   - fixed corrected split
   - same hyperparameters as the current paper
   - per-run metrics + mean ± SD

7) Cross-source novel-pattern generalization:
   - A -> residual B (all normalized overlaps with A removed)
   - B -> residual A (all normalized overlaps with B removed)
   - reports class imbalance explicitly
   - metrics include F1, PR-AUC, balanced accuracy, MCC
   - default: CodeBERT, seed 42
   - can be expanded to 5 seeds / both transformers via config

Outputs:
  /kaggle/working/sqlshield_corrected_validation/
    dataset_audit.csv
    conflict_groups.csv
    split_audit.csv
    classical_results.csv
    transformer_multiseed_runs.csv
    transformer_multiseed_summary.csv
    cross_source_runs.csv
    cross_source_summary.csv
    cross_source_overlap.csv
    predictions/*.csv
    histories/*.csv
    config.json
    results.json
"""

from __future__ import annotations

import gc
import json
import os
import random
import re
import time
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    set_seed as hf_set_seed,
)

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================

A_PATH = Path(
    "/kaggle/input/datasets/sajid576/sql-injection-dataset/"
    "Modified_SQL_Dataset.csv"
)
B_PATH = Path(
    "/kaggle/input/datasets/syedsaqlainhussain/sql-injection-dataset/"
    "sqliv2.csv"
)

A_NAME = "sajid576/sql-injection-dataset"
B_NAME = "syedsaqlainhussain/sql-injection-dataset/sqliv2.csv"

SPLIT_SEED = 42
SEEDS = [7, 21, 42, 84, 126]

MODELS = {
    "CodeBERT": "microsoft/codebert-base",
    "BERT-base": "bert-base-uncased",
}

# Cross-source default is intentionally lighter.
# To run 5 seeds externally too, set CROSS_SEEDS = SEEDS.
# To run BERT-base externally too, set CROSS_MODELS = MODELS.
CROSS_MODELS = {
    "CodeBERT": "microsoft/codebert-base",
}
CROSS_SEEDS = [42]

MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRAD_CLIP = 1.0

TFIDF_MAX_FEATURES = 10000

NUM_WORKERS = 0
PIN_MEMORY = True

OUT = Path("/kaggle/working/sqlshield_corrected_validation")
PRED_DIR = OUT / "predictions"
HIST_DIR = OUT / "histories"
CKPT_DIR = OUT / "checkpoints"

for d in [OUT, PRED_DIR, HIST_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# GENERAL UTILITIES
# ============================================================

def banner(s: str) -> None:
    print("\n" + "=" * 84)
    print(s)
    print("=" * 84)


def set_all_seeds(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    hf_set_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


def normalize_text(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


def read_b_csv(path: Path) -> pd.DataFrame:
    # sqliv2.csv is UTF-16 in the currently mounted Kaggle source.
    return pd.read_csv(path, encoding="utf-16", on_bad_lines="skip")


def strict_clean(
    raw: pd.DataFrame,
    text_col: str,
    label_col: str,
    source_name: str,
) -> pd.DataFrame:
    df = raw[[text_col, label_col]].copy()
    df.columns = ["text", "label"]

    df["text"] = df["text"].astype("string").str.strip()
    df["label"] = pd.to_numeric(df["label"], errors="coerce")

    df = df.dropna(subset=["text", "label"])
    df = df[df["label"].isin([0, 1])]
    df = df[df["text"].str.len() > 2].copy()

    df["label"] = df["label"].astype(int)
    df["source"] = source_name

    # Exact source-internal duplicate removal.
    df = df.drop_duplicates(
    subset=["text", "label"],
    keep="first"
).reset_index(drop=True)
    df["normalized_text"] = df["text"].map(normalize_text)

    return df


def load_sources() -> Tuple[pd.DataFrame, pd.DataFrame]:
    if not A_PATH.exists():
        raise FileNotFoundError(f"Dataset A not found: {A_PATH}")
    if not B_PATH.exists():
        raise FileNotFoundError(f"Dataset B not found: {B_PATH}")

    raw_a = pd.read_csv(A_PATH, on_bad_lines="skip")
    raw_b = read_b_csv(B_PATH)

    if "Query" not in raw_a.columns or "Label" not in raw_a.columns:
        raise ValueError(f"Unexpected A columns: {list(raw_a.columns)}")
    if "Sentence" not in raw_b.columns or "Label" not in raw_b.columns:
        raise ValueError(f"Unexpected B columns: {list(raw_b.columns)}")

    a = strict_clean(raw_a, "Query", "Label", A_NAME)
    b = strict_clean(raw_b, "Sentence", "Label", B_NAME)
    return a, b


def source_audit(df: pd.DataFrame, name: str) -> Dict:
    return {
        "dataset": name,
        "rows": int(len(df)),
        "benign": int((df["label"] == 0).sum()),
        "sqli": int((df["label"] == 1).sum()),
        "unique_exact_text": int(df["text"].nunique()),
        "unique_normalized_groups": int(df["normalized_text"].nunique()),
    }


def build_corrected_merged(
    a: pd.DataFrame,
    b: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Merge correctly normalized sources.

    Any normalized_text group with contradictory labels is removed completely.
    Then exact text duplicates across sources are removed.
    """
    combined = pd.concat([a, b], ignore_index=True)

    label_counts = combined.groupby("normalized_text")["label"].nunique()
    conflict_norms = set(label_counts[label_counts > 1].index)

    conflicts = combined[
        combined["normalized_text"].isin(conflict_norms)
    ].copy()

    clean = combined[
        ~combined["normalized_text"].isin(conflict_norms)
    ].copy()

    # Remove exact cross-source duplicates.
    clean = clean.drop_duplicates(subset=["text"], keep="first").reset_index(drop=True)

    # Safety: every normalized group must now have exactly one label.
    assert clean.groupby("normalized_text")["label"].nunique().max() == 1

    return clean, conflicts


# ============================================================
# GROUP-AWARE 80/10/10 SPLIT
# ============================================================

def make_group_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    One row per normalized group. Since conflicts were removed, each group has one label.
    """
    g = (
        df.groupby("normalized_text", as_index=False)
          .agg(
              label=("label", "first"),
              n_rows=("text", "size"),
          )
    )
    return g


def group_aware_split(
    df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Stratify UNIQUE NORMALIZED GROUPS by label, then map all group members
    into a single partition.

    This guarantees no normalized_text leakage across train/val/test.
    """
    groups = make_group_table(df)

    train_groups, temp_groups = train_test_split(
        groups,
        test_size=0.20,
        stratify=groups["label"],
        random_state=SPLIT_SEED,
    )

    val_groups, test_groups = train_test_split(
        temp_groups,
        test_size=0.50,
        stratify=temp_groups["label"],
        random_state=SPLIT_SEED,
    )

    train_set = set(train_groups["normalized_text"])
    val_set = set(val_groups["normalized_text"])
    test_set = set(test_groups["normalized_text"])

    assert train_set.isdisjoint(val_set)
    assert train_set.isdisjoint(test_set)
    assert val_set.isdisjoint(test_set)

    train_df = df[df["normalized_text"].isin(train_set)].copy().reset_index(drop=True)
    val_df = df[df["normalized_text"].isin(val_set)].copy().reset_index(drop=True)
    test_df = df[df["normalized_text"].isin(test_set)].copy().reset_index(drop=True)

    # Final zero-leakage assertions.
    assert set(train_df["normalized_text"]).isdisjoint(set(val_df["normalized_text"]))
    assert set(train_df["normalized_text"]).isdisjoint(set(test_df["normalized_text"]))
    assert set(val_df["normalized_text"]).isdisjoint(set(test_df["normalized_text"]))

    return train_df, val_df, test_df


def split_audit_rows(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> List[Dict]:
    rows = []
    total = len(train_df) + len(val_df) + len(test_df)

    for split_name, d in [
        ("train", train_df),
        ("validation", val_df),
        ("test", test_df),
    ]:
        rows.append({
            "split": split_name,
            "rows": int(len(d)),
            "row_pct": float(100 * len(d) / total),
            "benign": int((d["label"] == 0).sum()),
            "sqli": int((d["label"] == 1).sum()),
            "sqli_pct": float(100 * (d["label"] == 1).mean()),
            "normalized_groups": int(d["normalized_text"].nunique()),
        })

    return rows


# ============================================================
# METRICS
# ============================================================

@dataclass
class Metrics:
    accuracy: float
    precision: float
    recall: float
    f1: float
    roc_auc: float
    pr_auc: float
    balanced_accuracy: float
    mcc: float
    tn: int
    fp: int
    fn: int
    tp: int
    errors: int
    n: int


def compute_metrics(y_true, y_pred, y_score) -> Metrics:
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    y_score = np.asarray(y_score, dtype=float)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    try:
        roc = float(roc_auc_score(y_true, y_score))
    except Exception:
        roc = float("nan")

    try:
        pr = float(average_precision_score(y_true, y_score))
    except Exception:
        pr = float("nan")

    return Metrics(
        accuracy=float(accuracy_score(y_true, y_pred)),
        precision=float(precision_score(y_true, y_pred, zero_division=0)),
        recall=float(recall_score(y_true, y_pred, zero_division=0)),
        f1=float(f1_score(y_true, y_pred, zero_division=0)),
        roc_auc=roc,
        pr_auc=pr,
        balanced_accuracy=float(balanced_accuracy_score(y_true, y_pred)),
        mcc=float(matthews_corrcoef(y_true, y_pred)),
        tn=int(tn),
        fp=int(fp),
        fn=int(fn),
        tp=int(tp),
        errors=int((y_true != y_pred).sum()),
        n=int(len(y_true)),
    )


# ============================================================
# CLASSICAL BASELINES
# ============================================================

def classical_models() -> Dict[str, Pipeline]:
    return {
        "Random Forest": Pipeline([
            ("tfidf", TfidfVectorizer(
                max_features=TFIDF_MAX_FEATURES,
                ngram_range=(1, 2),
            )),
            ("clf", RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1,
            )),
        ]),
        "XGBoost": Pipeline([
            ("tfidf", TfidfVectorizer(
                max_features=TFIDF_MAX_FEATURES,
                ngram_range=(1, 2),
            )),
            ("clf", XGBClassifier(
                n_estimators=200,
                random_state=42,
                use_label_encoder=False,
                eval_metric="logloss",
            )),
        ]),
        "Logistic Regression": Pipeline([
            ("tfidf", TfidfVectorizer(
                max_features=TFIDF_MAX_FEATURES,
                ngram_range=(1, 2),
            )),
            ("clf", LogisticRegression(
                max_iter=1000,
                random_state=42,
            )),
        ]),
        "SVM": Pipeline([
            ("tfidf", TfidfVectorizer(
                max_features=TFIDF_MAX_FEATURES,
                ngram_range=(1, 2),
            )),
            ("clf", LinearSVC(
                max_iter=2000,
                random_state=42,
            )),
        ]),
    }


def run_classical(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> pd.DataFrame:
    banner("CLASSICAL BASELINES ON CORRECTED GROUP-AWARE SPLIT")

    rows = []

    for name, model in classical_models().items():
        print(f"\nTraining {name} ...")
        start = time.time()

        model.fit(train_df["text"], train_df["label"])
        preds = model.predict(test_df["text"])

        clf = model["clf"]
        if hasattr(clf, "predict_proba"):
            scores = model.predict_proba(test_df["text"])[:, 1]
        else:
            scores = model.decision_function(test_df["text"])

        metrics = compute_metrics(test_df["label"], preds, scores)
        row = {
            "model": name,
            **asdict(metrics),
            "elapsed_seconds": time.time() - start,
        }
        rows.append(row)

        pred_df = pd.DataFrame({
            "text": test_df["text"].values,
            "source": test_df["source"].values,
            "true_label": test_df["label"].values,
            "pred_label": preds,
            "score_sqli": scores,
            "correct": (preds == test_df["label"].values).astype(int),
        })
        pred_df.to_csv(
            PRED_DIR / f"classical__{name.replace(' ', '_')}.csv",
            index=False,
        )

        print(
            f"{name}: acc={metrics.accuracy:.6f}, "
            f"f1={metrics.f1:.6f}, roc_auc={metrics.roc_auc:.6f}, "
            f"errors={metrics.errors}, FP={metrics.fp}, FN={metrics.fn}"
        )

    out = pd.DataFrame(rows)
    out.to_csv(OUT / "classical_results.csv", index=False)
    return out


# ============================================================
# TRANSFORMER TRAINING
# ============================================================

class SQLDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer):
        self.texts = df["text"].tolist()
        self.labels = df["label"].astype(int).tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


def make_loader(
    df: pd.DataFrame,
    tokenizer,
    shuffle: bool,
    seed: int,
) -> DataLoader:
    gen = torch.Generator()
    gen.manual_seed(seed)

    return DataLoader(
        SQLDataset(df, tokenizer),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY and torch.cuda.is_available(),
        generator=gen if shuffle else None,
    )


def evaluate_transformer(model, loader):
    model.eval()

    preds_all, labels_all, probs_all = [], [], []

    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(DEVICE, non_blocking=True)
            mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
            labels = batch["labels"].to(DEVICE, non_blocking=True)

            out = model(input_ids=ids, attention_mask=mask)
            probs = torch.softmax(out.logits, dim=-1)[:, 1]
            preds = out.logits.argmax(dim=-1)

            preds_all.extend(preds.cpu().numpy().tolist())
            labels_all.extend(labels.cpu().numpy().tolist())
            probs_all.extend(probs.cpu().numpy().tolist())

    metrics = compute_metrics(labels_all, preds_all, probs_all)

    return (
        metrics,
        np.asarray(preds_all),
        np.asarray(labels_all),
        np.asarray(probs_all),
    )


def train_epoch(model, loader, optimizer, scheduler) -> float:
    model.train()
    total_loss = 0.0
    total_n = 0

    for batch in loader:
        ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        out = model(
            input_ids=ids,
            attention_mask=mask,
            labels=labels,
        )

        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        bs = labels.size(0)
        total_loss += float(out.loss.item()) * bs
        total_n += bs

    return total_loss / max(total_n, 1)


def train_transformer_once(
    experiment: str,
    model_name: str,
    model_id: str,
    seed: int,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> Dict:
    banner(
        f"{experiment} | {model_name} | seed={seed} | "
        f"train={len(train_df):,}, val={len(val_df):,}, test={len(test_df):,}"
    )

    set_all_seeds(seed)

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    train_loader = make_loader(train_df, tokenizer, True, seed)
    val_loader = make_loader(val_df, tokenizer, False, seed)
    test_loader = make_loader(test_df, tokenizer, False, seed)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=2,
        id2label={0: "benign", 1: "sqli"},
        label2id={"benign": 0, "sqli": 1},
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * WARMUP_RATIO),
        num_training_steps=total_steps,
    )

    safe_exp = re.sub(r"[^A-Za-z0-9_.-]+", "_", experiment)
    safe_model = re.sub(r"[^A-Za-z0-9_.-]+", "_", model_name)
    ckpt = CKPT_DIR / f"{safe_exp}__{safe_model}__seed{seed}.pt"

    best_val_f1 = -1.0
    best_epoch = None
    history = []
    start = time.time()

    for epoch in range(1, EPOCHS + 1):
        loss = train_epoch(model, train_loader, optimizer, scheduler)
        vm, _, _, _ = evaluate_transformer(model, val_loader)

        history.append({
            "epoch": epoch,
            "train_loss": loss,
            "val_accuracy": vm.accuracy,
            "val_precision": vm.precision,
            "val_recall": vm.recall,
            "val_f1": vm.f1,
            "val_roc_auc": vm.roc_auc,
            "val_pr_auc": vm.pr_auc,
        })

        print(
            f"Epoch {epoch}/{EPOCHS}: loss={loss:.6f}, "
            f"val_f1={vm.f1:.6f}, val_acc={vm.accuracy:.6f}"
        )

        if vm.f1 > best_val_f1:
            best_val_f1 = vm.f1
            best_epoch = epoch
            torch.save(model.state_dict(), ckpt)

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    tm, preds, labels, probs = evaluate_transformer(model, test_loader)

    elapsed = time.time() - start

    pred_df = pd.DataFrame({
        "text": test_df["text"].values,
        "source": test_df["source"].values,
        "true_label": labels,
        "pred_label": preds,
        "prob_sqli": probs,
        "correct": (preds == labels).astype(int),
    })
    pred_df.to_csv(
        PRED_DIR / f"{safe_exp}__{safe_model}__seed{seed}.csv",
        index=False,
    )

    pd.DataFrame(history).to_csv(
        HIST_DIR / f"{safe_exp}__{safe_model}__seed{seed}.csv",
        index=False,
    )

    result = {
        "experiment": experiment,
        "model": model_name,
        "model_id": model_id,
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_f1": best_val_f1,
        **asdict(tm),
        "elapsed_seconds": elapsed,
        "train_n": int(len(train_df)),
        "val_n": int(len(val_df)),
        "test_n": int(len(test_df)),
        "test_sqli": int((test_df["label"] == 1).sum()),
        "test_benign": int((test_df["label"] == 0).sum()),
    }

    print(
        f"TEST: acc={tm.accuracy:.6f}, f1={tm.f1:.6f}, "
        f"roc_auc={tm.roc_auc:.6f}, pr_auc={tm.pr_auc:.6f}, "
        f"bal_acc={tm.balanced_accuracy:.6f}, mcc={tm.mcc:.6f}, "
        f"errors={tm.errors}, FP={tm.fp}, FN={tm.fn}"
    )

    del model, tokenizer, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


SUMMARY_METRICS = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "balanced_accuracy",
    "mcc",
    "errors",
    "fp",
    "fn",
]


def summarize_runs(df: pd.DataFrame, group_cols: List[str]) -> pd.DataFrame:
    rows = []

    for keys, g in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = dict(zip(group_cols, keys))
        row["n_runs"] = int(len(g))

        for m in SUMMARY_METRICS:
            v = pd.to_numeric(g[m], errors="coerce")
            row[f"{m}_mean"] = float(v.mean())
            row[f"{m}_sd"] = float(v.std(ddof=1)) if len(v) > 1 else float("nan")
            row[f"{m}_min"] = float(v.min())
            row[f"{m}_max"] = float(v.max())

        rows.append(row)

    return pd.DataFrame(rows)


def run_transformer_multiseed(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rows = []

    for model_name, model_id in MODELS.items():
        for seed in SEEDS:
            rows.append(
                train_transformer_once(
                    experiment="corrected_group_split_multiseed",
                    model_name=model_name,
                    model_id=model_id,
                    seed=seed,
                    train_df=train_df,
                    val_df=val_df,
                    test_df=test_df,
                )
            )

    runs = pd.DataFrame(rows)
    summary = summarize_runs(runs, ["experiment", "model"])

    runs.to_csv(OUT / "transformer_multiseed_runs.csv", index=False)
    summary.to_csv(OUT / "transformer_multiseed_summary.csv", index=False)

    return runs, summary


# ============================================================
# CROSS-SOURCE NOVEL-PATTERN GENERALIZATION
# ============================================================

def residual_external(
    train_source: pd.DataFrame,
    external_source: pd.DataFrame,
) -> Tuple[pd.DataFrame, Dict]:
    train_norms = set(train_source["normalized_text"])
    mask = external_source["normalized_text"].isin(train_norms)

    ext = external_source.loc[~mask].copy().reset_index(drop=True)

    report = {
        "external_before": int(len(external_source)),
        "removed_normalized_overlap": int(mask.sum()),
        "external_after": int(len(ext)),
        "external_remaining_pct": float(100 * len(ext) / len(external_source)),
        "benign_after": int((ext["label"] == 0).sum()),
        "sqli_after": int((ext["label"] == 1).sum()),
        "sqli_pct_after": float(100 * (ext["label"] == 1).mean()),
    }

    if ext["label"].nunique() < 2:
        raise ValueError("Residual external set has only one class.")

    return ext, report


def source_train_val(source_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Group-aware train/val split inside ONE source.
    """
    groups = make_group_table(source_df)

    tr_g, va_g = train_test_split(
        groups,
        test_size=0.10,
        stratify=groups["label"],
        random_state=SPLIT_SEED,
    )

    tr_set = set(tr_g["normalized_text"])
    va_set = set(va_g["normalized_text"])

    train_df = source_df[
        source_df["normalized_text"].isin(tr_set)
    ].copy().reset_index(drop=True)

    val_df = source_df[
        source_df["normalized_text"].isin(va_set)
    ].copy().reset_index(drop=True)

    assert set(train_df["normalized_text"]).isdisjoint(
        set(val_df["normalized_text"])
    )

    return train_df, val_df


def run_cross_direction(
    train_source: pd.DataFrame,
    external_source: pd.DataFrame,
    train_name: str,
    external_name: str,
) -> Tuple[List[Dict], Dict]:
    ext, overlap = residual_external(train_source, external_source)
    train_df, val_df = source_train_val(train_source)

    experiment = f"cross_source_{train_name}_TO_{external_name}"

    overlap.update({
        "experiment": experiment,
        "train_source": train_name,
        "external_source": external_name,
        "train_source_rows": int(len(train_source)),
        "train_partition_rows": int(len(train_df)),
        "val_partition_rows": int(len(val_df)),
    })

    banner(experiment)
    print(json.dumps(overlap, indent=2))

    rows = []

    for model_name, model_id in CROSS_MODELS.items():
        for seed in CROSS_SEEDS:
            r = train_transformer_once(
                experiment=experiment,
                model_name=model_name,
                model_id=model_id,
                seed=seed,
                train_df=train_df,
                val_df=val_df,
                test_df=ext,
            )
            r.update({
                "train_source": train_name,
                "external_source": external_name,
                **overlap,
            })
            rows.append(r)

    return rows, overlap


def run_cross_source(
    a: pd.DataFrame,
    b: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rows = []
    overlap_rows = []

    ab, ab_rep = run_cross_direction(
        a, b,
        "A_sajid576",
        "B_sqliv2",
    )
    rows.extend(ab)
    overlap_rows.append(ab_rep)

    ba, ba_rep = run_cross_direction(
        b, a,
        "B_sqliv2",
        "A_sajid576",
    )
    rows.extend(ba)
    overlap_rows.append(ba_rep)

    runs = pd.DataFrame(rows)
    overlap_df = pd.DataFrame(overlap_rows)
    summary = summarize_runs(
        runs,
        ["experiment", "model", "train_source", "external_source"],
    )

    runs.to_csv(OUT / "cross_source_runs.csv", index=False)
    overlap_df.to_csv(OUT / "cross_source_overlap.csv", index=False)
    summary.to_csv(OUT / "cross_source_summary.csv", index=False)

    return runs, summary, overlap_df


# ============================================================
# MAIN
# ============================================================

def main():
    banner("SQLShield Corrected Validation Pipeline")

    print(f"Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"GPU count: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

    # --------------------------------------------------------
    # Load sources correctly
    # --------------------------------------------------------
    a, b = load_sources()

    audit = pd.DataFrame([
        source_audit(a, "A_sajid576"),
        source_audit(b, "B_sqliv2"),
    ])

    # Source overlap audit
    exact_overlap = len(set(a["text"]) & set(b["text"]))
    norm_overlap = len(
        set(a["normalized_text"]) & set(b["normalized_text"])
    )

    banner("SOURCE AUDIT")
    print(audit.to_string(index=False))
    print(f"\nExact text overlap: {exact_overlap:,}")
    print(f"Normalized overlap groups: {norm_overlap:,}")

    # --------------------------------------------------------
    # Corrected merged corpus
    # --------------------------------------------------------
    merged, conflicts = build_corrected_merged(a, b)

    conflicts.to_csv(OUT / "conflict_groups.csv", index=False)

    print(f"\nConflicting normalized groups removed: "
          f"{conflicts['normalized_text'].nunique() if len(conflicts) else 0}")
    print(f"Rows removed due to conflicting normalized labels: {len(conflicts)}")

    merged_audit = {
        "dataset": "corrected_merged",
        "rows": int(len(merged)),
        "benign": int((merged["label"] == 0).sum()),
        "sqli": int((merged["label"] == 1).sum()),
        "unique_exact_text": int(merged["text"].nunique()),
        "unique_normalized_groups": int(merged["normalized_text"].nunique()),
        "exact_source_overlap_before_merge": int(exact_overlap),
        "normalized_source_overlap_before_merge": int(norm_overlap),
        "conflicting_normalized_groups_removed": int(
            conflicts["normalized_text"].nunique() if len(conflicts) else 0
        ),
        "conflicting_rows_removed": int(len(conflicts)),
    }

    audit = pd.concat(
        [audit, pd.DataFrame([merged_audit])],
        ignore_index=True,
    )
    audit.to_csv(OUT / "dataset_audit.csv", index=False)

    banner("CORRECTED MERGED CORPUS")
    print(json.dumps(merged_audit, indent=2))

    # --------------------------------------------------------
    # Group-aware fixed split
    # --------------------------------------------------------
    train_df, val_df, test_df = group_aware_split(merged)

    split_rows = split_audit_rows(train_df, val_df, test_df)
    split_df = pd.DataFrame(split_rows)
    split_df.to_csv(OUT / "split_audit.csv", index=False)

    banner("GROUP-AWARE SPLIT AUDIT")
    print(split_df.to_string(index=False))

    # Explicit zero-overlap report
    overlap_checks = {
        "train_val_norm_overlap": len(
            set(train_df["normalized_text"]) & set(val_df["normalized_text"])
        ),
        "train_test_norm_overlap": len(
            set(train_df["normalized_text"]) & set(test_df["normalized_text"])
        ),
        "val_test_norm_overlap": len(
            set(val_df["normalized_text"]) & set(test_df["normalized_text"])
        ),
    }
    print("\nNormalized-group overlap across partitions:")
    print(json.dumps(overlap_checks, indent=2))

    assert all(v == 0 for v in overlap_checks.values())

    # Save split IDs/text for exact reproducibility.
    train_df.to_csv(OUT / "train_corrected.csv", index=False)
    val_df.to_csv(OUT / "val_corrected.csv", index=False)
    test_df.to_csv(OUT / "test_corrected.csv", index=False)

    # --------------------------------------------------------
    # Config snapshot
    # --------------------------------------------------------
    config = {
        "A_PATH": str(A_PATH),
        "B_PATH": str(B_PATH),
        "A_NAME": A_NAME,
        "B_NAME": B_NAME,
        "split_seed": SPLIT_SEED,
        "training_seeds": SEEDS,
        "models": MODELS,
        "cross_models": CROSS_MODELS,
        "cross_seeds": CROSS_SEEDS,
        "max_len": MAX_LEN,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "grad_clip": GRAD_CLIP,
        "tfidf_max_features": TFIDF_MAX_FEATURES,
        "normalization": "strip + lowercase + collapse internal whitespace",
        "conflict_policy": "remove normalized groups with contradictory labels",
        "split_policy": (
            "stratified split of unique normalized groups; "
            "all rows from each normalized group remain in one partition"
        ),
        "overlap_checks": overlap_checks,
        "device": str(DEVICE),
    }

    (OUT / "config.json").write_text(
        json.dumps(config, indent=2),
        encoding="utf-8",
    )

    # --------------------------------------------------------
    # Classical baselines
    # --------------------------------------------------------
    classical = run_classical(train_df, test_df)

    # --------------------------------------------------------
    # Transformer 5-seed stability
    # --------------------------------------------------------
    multiseed_runs, multiseed_summary = run_transformer_multiseed(
        train_df, val_df, test_df
    )

    # --------------------------------------------------------
    # Cross-source residual generalization
    # --------------------------------------------------------
    cross_runs, cross_summary, cross_overlap = run_cross_source(a, b)

    # --------------------------------------------------------
    # Consolidated JSON
    # --------------------------------------------------------
    results = {
        "config": config,
        "dataset_audit": audit.to_dict(orient="records"),
        "split_audit": split_rows,
        "classical_results": classical.to_dict(orient="records"),
        "transformer_multiseed_runs": multiseed_runs.to_dict(orient="records"),
        "transformer_multiseed_summary": multiseed_summary.to_dict(orient="records"),
        "cross_source_runs": cross_runs.to_dict(orient="records"),
        "cross_source_summary": cross_summary.to_dict(orient="records"),
        "cross_source_overlap": cross_overlap.to_dict(orient="records"),
    }

    (OUT / "results.json").write_text(
        json.dumps(results, indent=2, default=str),
        encoding="utf-8",
    )

    banner("ALL EXPERIMENTS COMPLETE")
    print("\nTransformer multi-seed summary:")
    print(multiseed_summary.to_string(index=False))

    print("\nCross-source summary:")
    print(cross_summary.to_string(index=False))

    print("\nImportant output files:")
    for name in [
        "dataset_audit.csv",
        "split_audit.csv",
        "classical_results.csv",
        "transformer_multiseed_runs.csv",
        "transformer_multiseed_summary.csv",
        "cross_source_runs.csv",
        "cross_source_summary.csv",
        "cross_source_overlap.csv",
        "results.json",
    ]:
        print(" ", OUT / name)

    print(
        "\nNOTE: The residual cross-source external sets are highly imbalanced "
        "(few SQLi samples after normalized-overlap removal). Interpret PR-AUC, "
        "SQLi recall/F1, balanced accuracy, MCC, FP, and FN alongside accuracy."
    )


#if __name__ == "__main__":
  #  main()

In [5]:
a, b = load_sources()
merged, conflicts = build_corrected_merged(a, b)
train_df, val_df, test_df = group_aware_split(merged)

print(len(train_df), len(val_df), len(test_df))
print(
    len(set(train_df["normalized_text"]) & set(test_df["normalized_text"]))
)

45288 5679 5654
0


In [14]:
print("DEVICE:", DEVICE)

# Load the two datasets correctly
a, b = load_sources()

print("\n=== SOURCE A ===")
print("Rows:", len(a))
print("Benign:", (a["label"] == 0).sum())
print("SQLi:", (a["label"] == 1).sum())
print("Normalized groups:", a["normalized_text"].nunique())

print("\n=== SOURCE B ===")
print("Rows:", len(b))
print("Benign:", (b["label"] == 0).sum())
print("SQLi:", (b["label"] == 1).sum())
print("Normalized groups:", b["normalized_text"].nunique())

# Build corrected merged corpus
merged, conflicts = build_corrected_merged(a, b)

print("\n=== CORRECTED MERGED DATASET ===")
print("Rows:", len(merged))
print("Benign:", (merged["label"] == 0).sum())
print("SQLi:", (merged["label"] == 1).sum())
print("Normalized groups:", merged["normalized_text"].nunique())

print("\n=== CONFLICTS REMOVED ===")
print("Conflict rows:", len(conflicts))
print(
    "Conflict groups:",
    conflicts["normalized_text"].nunique() if len(conflicts) else 0
)

# Group-aware split
train_df, val_df, test_df = group_aware_split(merged)

print("\n=== GROUP-AWARE SPLIT ===")

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(
        name,
        "| rows =", len(df),
        "| benign =", (df["label"] == 0).sum(),
        "| SQLi =", (df["label"] == 1).sum(),
        "| SQLi % =", round((df["label"] == 1).mean() * 100, 2),
        "| groups =", df["normalized_text"].nunique()
    )

print("\n=== LEAKAGE CHECK ===")

print(
    "Train-Val overlap:",
    len(
        set(train_df["normalized_text"])
        & set(val_df["normalized_text"])
    )
)

print(
    "Train-Test overlap:",
    len(
        set(train_df["normalized_text"])
        & set(test_df["normalized_text"])
    )
)

print(
    "Val-Test overlap:",
    len(
        set(val_df["normalized_text"])
        & set(test_df["normalized_text"])
    )
)

DEVICE: cuda

=== SOURCE A ===
Rows: 30766
Benign: 19481
SQLi: 11285
Normalized groups: 30736

=== SOURCE B ===
Rows: 33537
Benign: 22171
SQLi: 11366
Normalized groups: 33527

=== CORRECTED MERGED DATASET ===
Rows: 56621
Benign: 34490
SQLi: 22131
Normalized groups: 45915

=== CONFLICTS REMOVED ===
Conflict rows: 2
Conflict groups: 1

=== GROUP-AWARE SPLIT ===
Train | rows = 45288 | benign = 27589 | SQLi = 17699 | SQLi % = 39.08 | groups = 36732
Validation | rows = 5679 | benign = 3451 | SQLi = 2228 | SQLi % = 39.23 | groups = 4591
Test | rows = 5654 | benign = 3450 | SQLi = 2204 | SQLi % = 38.98 | groups = 4592

=== LEAKAGE CHECK ===
Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


In [15]:
classical_results = run_classical(
    train_df,
    test_df
)

print("\n=== CLASSICAL RESULTS ===")
print(
    classical_results[
        [
            "model",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "balanced_accuracy",
            "mcc",
            "errors",
            "fp",
            "fn"
        ]
    ]
    .sort_values("f1", ascending=False)
    .to_string(index=False)
)


CLASSICAL BASELINES ON CORRECTED GROUP-AWARE SPLIT

Training Random Forest ...
Random Forest: acc=0.997170, f1=0.996365, roc_auc=0.998884, errors=16, FP=5, FN=11

Training XGBoost ...
XGBoost: acc=0.996286, f1=0.995224, roc_auc=0.998267, errors=21, FP=5, FN=16

Training Logistic Regression ...
Logistic Regression: acc=0.990803, f1=0.988144, roc_auc=0.998396, errors=52, FP=15, FN=37

Training SVM ...
SVM: acc=0.995578, f1=0.994322, roc_auc=0.998418, errors=25, FP=10, FN=15

=== CLASSICAL RESULTS ===
              model  accuracy  precision   recall       f1  roc_auc   pr_auc  balanced_accuracy      mcc  errors  fp  fn
      Random Forest  0.997170   0.997725 0.995009 0.996365 0.998884 0.998386           0.996780 0.994051      16   5  11
            XGBoost  0.996286   0.997720 0.992740 0.995224 0.998267 0.997197           0.995646 0.992194      21   5  16
                SVM  0.995578   0.995452 0.993194 0.994322 0.998418 0.997839           0.995148 0.990703      25  10  15
Logistic Re

In [16]:
codebert_42 = train_transformer_once(
    experiment="corrected_group_split_pilot",
    model_name="CodeBERT",
    model_id="microsoft/codebert-base",
    seed=42,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
)

print("\n=== CODEBERT SEED 42 ===")

for k, v in codebert_42.items():
    print(f"{k}: {v}")


corrected_group_split_pilot | CodeBERT | seed=42 | train=45,288, val=5,679, test=5,654


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Epoch 1/4: loss=0.058602, val_f1=0.999551, val_acc=0.999648
Epoch 2/4: loss=0.002434, val_f1=0.998655, val_acc=0.998943
Epoch 3/4: loss=0.002158, val_f1=0.999102, val_acc=0.999296
Epoch 4/4: loss=0.000423, val_f1=0.999102, val_acc=0.999296
TEST: acc=0.999469, f1=0.999319, roc_auc=0.999971, pr_auc=0.999959, bal_acc=0.999401, mcc=0.998885, errors=3, FP=1, FN=2

=== CODEBERT SEED 42 ===
experiment: corrected_group_split_pilot
model: CodeBERT
model_id: microsoft/codebert-base
seed: 42
best_epoch: 1
best_val_f1: 0.9995509654243376
accuracy: 0.9994694021931376
precision: 0.9995460735360872
recall: 0.9990925589836661
f1: 0.9993192648059904
roc_auc: 0.9999710670980299
pr_auc: 0.9999587498697292
balanced_accuracy: 0.9994013519556011
mcc: 0.9988846142841227
tn: 3449
fp: 1
fn: 2
tp: 2202
errors: 3
n: 5654
elapsed_seconds: 4173.960512876511
train_n: 45288
val_n: 5679
test_n: 5654
test_sqli: 2204
test_benign: 3450


In [17]:
bert_42 = train_transformer_once(
    experiment="corrected_group_split_pilot",
    model_name="BERT-base",
    model_id="bert-base-uncased",
    seed=42,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
)

print("\n=== BERT-BASE SEED 42 ===")

for k, v in bert_42.items():
    print(f"{k}: {v}")


corrected_group_split_pilot | BERT-base | seed=42 | train=45,288, val=5,679, test=5,654


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/4: loss=0.063924, val_f1=0.997981, val_acc=0.998415
Epoch 2/4: loss=0.003744, val_f1=0.997314, val_acc=0.997887
Epoch 3/4: loss=0.000914, val_f1=0.999776, val_acc=0.999824
Epoch 4/4: loss=0.000480, val_f1=0.999326, val_acc=0.999472
TEST: acc=0.998939, f1=0.998638, roc_auc=0.999955, pr_auc=0.999932, bal_acc=0.998803, mcc=0.997769, errors=6, FP=2, FN=4

=== BERT-BASE SEED 42 ===
experiment: corrected_group_split_pilot
model: BERT-base
model_id: bert-base-uncased
seed: 42
best_epoch: 3
best_val_f1: 0.9997755331088665
accuracy: 0.9989388043862752
precision: 0.9990917347865577
recall: 0.9981851179673321
f1: 0.9986382206082615
roc_auc: 0.9999546279491833
pr_auc: 0.9999319733075356
balanced_accuracy: 0.9988027039112023
mcc: 0.9977691835852571
tn: 3448
fp: 2
fn: 4
tp: 2200
errors: 6
n: 5654
elapsed_seconds: 4158.610219955444
train_n: 45288
val_n: 5679
test_n: 5654
test_sqli: 2204
test_benign: 3450


In [2]:
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

print("Folder exists:", SAVE_DIR.exists())

if SAVE_DIR.exists():
    for f in sorted(SAVE_DIR.rglob("*")):
        if f.is_file():
            print(f)

Folder exists: True


In [3]:
from pathlib import Path

search_roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]

keywords = [
    "CodeBERT",
    "BERT-base",
    "classical_results",
    "Random_Forest",
    "seed42",
    "mcnemar",
]

found = []

for root in search_roots:
    if root.exists():
        for f in root.rglob("*"):
            if f.is_file():
                name = str(f)
                if any(k.lower() in name.lower() for k in keywords):
                    found.append(name)

print("Found files:", len(found))
for x in found:
    print(x)

Found files: 0


In [6]:
a, b = load_sources()
merged, conflicts = build_corrected_merged(a, b)
train_df, val_df, test_df = group_aware_split(merged)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print(
    "Train-Test overlap:",
    len(
        set(train_df["normalized_text"]) &
        set(test_df["normalized_text"])
    )
)

Train: 45288
Validation: 5679
Test: 5654
Train-Test overlap: 0


In [7]:
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(SAVE_DIR / "train_corrected.csv", index=False)
val_df.to_csv(SAVE_DIR / "val_corrected.csv", index=False)
test_df.to_csv(SAVE_DIR / "test_corrected.csv", index=False)

print("Saved successfully:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

for f in SAVE_DIR.glob("*.csv"):
    print(f)

Saved successfully:
Train: 45288
Validation: 5679
Test: 5654
/kaggle/working/sqlshield_corrected_validation/train_corrected.csv
/kaggle/working/sqlshield_corrected_validation/test_corrected.csv
/kaggle/working/sqlshield_corrected_validation/val_corrected.csv


In [8]:
import pandas as pd
import json
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")
PRED_DIR = SAVE_DIR / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

# Random Forest only
rf_model = classical_models()["Random Forest"]

print("Training Random Forest...")
rf_model.fit(
    train_df["text"],
    train_df["label"]
)

rf_pred = rf_model.predict(test_df["text"])
rf_score = rf_model.predict_proba(test_df["text"])[:, 1]

rf_metrics = compute_metrics(
    test_df["label"],
    rf_pred,
    rf_score
)

print("\n=== RANDOM FOREST ===")
for k, v in rf_metrics.__dict__.items():
    print(f"{k}: {v}")

# Save predictions
rf_predictions = pd.DataFrame({
    "text": test_df["text"].values,
    "source": test_df["source"].values,
    "true_label": test_df["label"].values,
    "pred_label": rf_pred,
    "score_sqli": rf_score,
    "correct": (
        rf_pred == test_df["label"].values
    ).astype(int)
})

rf_predictions.to_csv(
    PRED_DIR / "classical__Random_Forest.csv",
    index=False
)

# Save metrics
with open(
    SAVE_DIR / "random_forest_result.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        rf_metrics.__dict__,
        f,
        indent=2,
        default=str
    )

print("\nSaved:")
print(PRED_DIR / "classical__Random_Forest.csv")
print(SAVE_DIR / "random_forest_result.json")

Training Random Forest...

=== RANDOM FOREST ===
accuracy: 0.9971701450300672
precision: 0.9977252047315741
recall: 0.9950090744101633
f1: 0.9963652885052249
roc_auc: 0.9988843736026723
pr_auc: 0.998385958713976
balanced_accuracy: 0.9967798995239222
mcc: 0.994050945310396
tn: 3445
fp: 5
fn: 11
tp: 2193
errors: 16
n: 5654

Saved:
/kaggle/working/sqlshield_corrected_validation/predictions/classical__Random_Forest.csv
/kaggle/working/sqlshield_corrected_validation/random_forest_result.json


In [9]:
codebert_42 = train_transformer_once(
    experiment="corrected_group_split_pilot",
    model_name="CodeBERT",
    model_id="microsoft/codebert-base",
    seed=42,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
)

print("\n=== CODEBERT SEED 42 ===")

for k, v in codebert_42.items():
    print(f"{k}: {v}")


corrected_group_split_pilot | CodeBERT | seed=42 | train=45,288, val=5,679, test=5,654


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Epoch 1/4: loss=0.058602, val_f1=0.999551, val_acc=0.999648
Epoch 2/4: loss=0.002434, val_f1=0.998655, val_acc=0.998943
Epoch 3/4: loss=0.002158, val_f1=0.999102, val_acc=0.999296
Epoch 4/4: loss=0.000423, val_f1=0.999102, val_acc=0.999296
TEST: acc=0.999469, f1=0.999319, roc_auc=0.999971, pr_auc=0.999959, bal_acc=0.999401, mcc=0.998885, errors=3, FP=1, FN=2

=== CODEBERT SEED 42 ===
experiment: corrected_group_split_pilot
model: CodeBERT
model_id: microsoft/codebert-base
seed: 42
best_epoch: 1
best_val_f1: 0.9995509654243376
accuracy: 0.9994694021931376
precision: 0.9995460735360872
recall: 0.9990925589836661
f1: 0.9993192648059904
roc_auc: 0.9999710670980299
pr_auc: 0.9999587498697292
balanced_accuracy: 0.9994013519556011
mcc: 0.9988846142841227
tn: 3449
fp: 1
fn: 2
tp: 2202
errors: 3
n: 5654
elapsed_seconds: 3911.2582082748413
train_n: 45288
val_n: 5679
test_n: 5654
test_sqli: 2204
test_benign: 3450


In [10]:
import json
import shutil
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

# Save CodeBERT metrics explicitly
with open(
    SAVE_DIR / "codebert_seed42_result.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        codebert_42,
        f,
        indent=2,
        default=str
    )

print("=== CURRENT FILES ===")
for f in sorted(SAVE_DIR.rglob("*")):
    if f.is_file():
        print(f)

# Create full backup ZIP
backup_path = shutil.make_archive(
    "/kaggle/working/sqlshield_after_codebert",
    "zip",
    SAVE_DIR
)

print("\nBackup created:")
print(backup_path)

=== CURRENT FILES ===
/kaggle/working/sqlshield_corrected_validation/checkpoints/corrected_group_split_pilot__CodeBERT__seed42.pt
/kaggle/working/sqlshield_corrected_validation/codebert_seed42_result.json
/kaggle/working/sqlshield_corrected_validation/histories/corrected_group_split_pilot__CodeBERT__seed42.csv
/kaggle/working/sqlshield_corrected_validation/predictions/classical__Random_Forest.csv
/kaggle/working/sqlshield_corrected_validation/predictions/corrected_group_split_pilot__CodeBERT__seed42.csv
/kaggle/working/sqlshield_corrected_validation/random_forest_result.json
/kaggle/working/sqlshield_corrected_validation/test_corrected.csv
/kaggle/working/sqlshield_corrected_validation/train_corrected.csv
/kaggle/working/sqlshield_corrected_validation/val_corrected.csv

Backup created:
/kaggle/working/sqlshield_after_codebert.zip


In [12]:
from IPython.display import FileLink, display

display(
    FileLink(
        "/kaggle/working/sqlshield_after_codebert.zip"
    )
)

/kaggle/working/sqlshield_after_codebert.zip

In [13]:
import os

path = "/kaggle/working/sqlshield_after_codebert.zip"

print("Exists:", os.path.exists(path))
print("Size MB:", round(os.path.getsize(path) / 1024**2, 2))

Exists: True
Size MB: 443.28


In [14]:
from IPython.display import HTML, display

display(HTML("""
<a href="files/sqlshield_after_codebert.zip"
   download
   style="font-size:18px;font-weight:bold;">
   اضغط هنا لتنزيل SQLShield Backup
</a>
"""))

In [15]:
bert_42 = train_transformer_once(
    experiment="corrected_group_split_pilot",
    model_name="BERT-base",
    model_id="bert-base-uncased",
    seed=42,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
)

print("\n=== BERT-BASE SEED 42 ===")

for k, v in bert_42.items():
    print(f"{k}: {v}")


corrected_group_split_pilot | BERT-base | seed=42 | train=45,288, val=5,679, test=5,654


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/4: loss=0.063924, val_f1=0.997981, val_acc=0.998415
Epoch 2/4: loss=0.003744, val_f1=0.997314, val_acc=0.997887
Epoch 3/4: loss=0.000914, val_f1=0.999776, val_acc=0.999824
Epoch 4/4: loss=0.000480, val_f1=0.999326, val_acc=0.999472
TEST: acc=0.998939, f1=0.998638, roc_auc=0.999955, pr_auc=0.999932, bal_acc=0.998803, mcc=0.997769, errors=6, FP=2, FN=4

=== BERT-BASE SEED 42 ===
experiment: corrected_group_split_pilot
model: BERT-base
model_id: bert-base-uncased
seed: 42
best_epoch: 3
best_val_f1: 0.9997755331088665
accuracy: 0.9989388043862752
precision: 0.9990917347865577
recall: 0.9981851179673321
f1: 0.9986382206082615
roc_auc: 0.9999546279491833
pr_auc: 0.9999319733075356
balanced_accuracy: 0.9988027039112023
mcc: 0.9977691835852571
tn: 3448
fp: 2
fn: 4
tp: 2200
errors: 6
n: 5654
elapsed_seconds: 3902.9999306201935
train_n: 45288
val_n: 5679
test_n: 5654
test_sqli: 2204
test_benign: 3450


In [16]:
import json
import shutil
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

# Save BERT metrics
with open(
    SAVE_DIR / "bert_seed42_result.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        bert_42,
        f,
        indent=2,
        default=str
    )

# Show current files
print("=== CURRENT FILES ===")
for f in sorted(SAVE_DIR.rglob("*")):
    if f.is_file():
        print(f)

# Full backup after BERT
backup_path = shutil.make_archive(
    "/kaggle/working/sqlshield_after_bert",
    "zip",
    SAVE_DIR
)

print("\nBackup created:")
print(backup_path)

=== CURRENT FILES ===
/kaggle/working/sqlshield_corrected_validation/bert_seed42_result.json
/kaggle/working/sqlshield_corrected_validation/checkpoints/corrected_group_split_pilot__BERT-base__seed42.pt
/kaggle/working/sqlshield_corrected_validation/checkpoints/corrected_group_split_pilot__CodeBERT__seed42.pt
/kaggle/working/sqlshield_corrected_validation/codebert_seed42_result.json
/kaggle/working/sqlshield_corrected_validation/histories/corrected_group_split_pilot__BERT-base__seed42.csv
/kaggle/working/sqlshield_corrected_validation/histories/corrected_group_split_pilot__CodeBERT__seed42.csv
/kaggle/working/sqlshield_corrected_validation/predictions/classical__Random_Forest.csv
/kaggle/working/sqlshield_corrected_validation/predictions/corrected_group_split_pilot__BERT-base__seed42.csv
/kaggle/working/sqlshield_corrected_validation/predictions/corrected_group_split_pilot__CodeBERT__seed42.csv
/kaggle/working/sqlshield_corrected_validation/random_forest_result.json
/kaggle/working/sqls

In [18]:
from pathlib import Path
import pandas as pd
from scipy.stats import binomtest

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")
PRED_DIR = SAVE_DIR / "predictions"

# Load predictions
cb = pd.read_csv(
    PRED_DIR / "corrected_group_split_pilot__CodeBERT__seed42.csv"
)

bert = pd.read_csv(
    PRED_DIR / "corrected_group_split_pilot__BERT-base__seed42.csv"
)

rf = pd.read_csv(
    PRED_DIR / "classical__Random_Forest.csv"
)

# Verify all models used exactly the same test set
assert len(cb) == len(bert) == len(rf) == 5654
assert (cb["text"].values == bert["text"].values).all()
assert (cb["text"].values == rf["text"].values).all()

assert (
    cb["true_label"].values ==
    bert["true_label"].values
).all()

assert (
    cb["true_label"].values ==
    rf["true_label"].values
).all()

def exact_mcnemar(df1, df2, name1, name2):

    correct1 = (
        df1["pred_label"].values ==
        df1["true_label"].values
    )

    correct2 = (
        df2["pred_label"].values ==
        df2["true_label"].values
    )

    # Model 1 wrong, Model 2 correct
    b = int((~correct1 & correct2).sum())

    # Model 1 correct, Model 2 wrong
    c = int((correct1 & ~correct2).sum())

    n = b + c

    if n == 0:
        p = 1.0
    else:
        p = binomtest(
            min(b, c),
            n=n,
            p=0.5,
            alternative="two-sided"
        ).pvalue

    return {
        "comparison": f"{name1} vs {name2}",
        "model1_wrong_model2_correct": b,
        "model1_correct_model2_wrong": c,
        "discordant_pairs": n,
        "exact_p_value": p
    }

results = [
    exact_mcnemar(cb, bert, "CodeBERT", "BERT-base"),
    exact_mcnemar(cb, rf, "CodeBERT", "Random Forest"),
    exact_mcnemar(bert, rf, "BERT-base", "Random Forest"),
]

mcnemar_df = pd.DataFrame(results)

print("\n=== EXACT McNEMAR RESULTS ===")
print(mcnemar_df.to_string(index=False))

mcnemar_df.to_csv(
    SAVE_DIR / "pilot_exact_mcnemar.csv",
    index=False
)

print("\nSaved:")
print(SAVE_DIR / "pilot_exact_mcnemar.csv")


=== EXACT McNEMAR RESULTS ===
                comparison  model1_wrong_model2_correct  model1_correct_model2_wrong  discordant_pairs  exact_p_value
     CodeBERT vs BERT-base                            2                            5                 7       0.453125
 CodeBERT vs Random Forest                            2                           15                17       0.002350
BERT-base vs Random Forest                            2                           12                14       0.012939

Saved:
/kaggle/working/sqlshield_corrected_validation/pilot_exact_mcnemar.csv


In [ ]:
from pathlib import Path
import pandas as pd
import json
import gc
import torch

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

# Reload exact saved splits
train_df = pd.read_csv(SAVE_DIR / "train_corrected.csv")
val_df   = pd.read_csv(SAVE_DIR / "val_corrected.csv")
test_df  = pd.read_csv(SAVE_DIR / "test_corrected.csv")

# Existing seed-42 results
with open(SAVE_DIR / "codebert_seed42_result.json", "r") as f:
    codebert_42_saved = json.load(f)

with open(SAVE_DIR / "bert_seed42_result.json", "r") as f:
    bert_42_saved = json.load(f)

# We already completed seed 42
remaining_seeds = [7, 21, 84, 126]

models_to_run = {
    "CodeBERT": "microsoft/codebert-base",
    "BERT-base": "bert-base-uncased",
}

# Start table with already completed seed 42
all_results = []

cb42 = dict(codebert_42_saved)
cb42["experiment"] = "corrected_group_split_multiseed"
all_results.append(cb42)

b42 = dict(bert_42_saved)
b42["experiment"] = "corrected_group_split_multiseed"
all_results.append(b42)

RESULT_FILE = SAVE_DIR / "transformer_multiseed_runs.csv"

# If a partial results file already exists, use it
if RESULT_FILE.exists():
    old = pd.read_csv(RESULT_FILE)

    existing_keys = set(
        zip(old["model"], old["seed"])
    )

    for row in old.to_dict("records"):
        key = (row["model"], int(row["seed"]))

        if key not in {
            (r["model"], int(r["seed"]))
            for r in all_results
        }:
            all_results.append(row)
else:
    existing_keys = {
        ("CodeBERT", 42),
        ("BERT-base", 42),
    }


for model_name, model_id in models_to_run.items():

    for seed in remaining_seeds:

        key = (model_name, seed)

        # Resume-safe
        completed = {
            (r["model"], int(r["seed"]))
            for r in all_results
        }

        if key in completed:
            print(f"SKIP: {model_name} seed={seed} already completed")
            continue

        print("\n" + "=" * 80)
        print(f"RUNNING: {model_name} | seed={seed}")
        print("=" * 80)

        result = train_transformer_once(
            experiment="corrected_group_split_multiseed",
            model_name=model_name,
            model_id=model_id,
            seed=seed,
            train_df=train_df,
            val_df=val_df,
            test_df=test_df,
        )

        all_results.append(result)

        # SAVE IMMEDIATELY AFTER EACH SEED
        runs_df = pd.DataFrame(all_results)

        runs_df.to_csv(
            RESULT_FILE,
            index=False
        )

        # Separate JSON backup
        with open(
            SAVE_DIR / "transformer_multiseed_runs.json",
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                all_results,
                f,
                indent=2,
                default=str
            )

        print("\nSAVED AFTER THIS SEED:")
        print(RESULT_FILE)

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ============================================================
# FINAL SUMMARY
# ============================================================

runs_df = pd.DataFrame(all_results)

metrics = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "balanced_accuracy",
    "mcc",
    "errors",
    "fp",
    "fn",
]

summary_rows = []

for model_name, group in runs_df.groupby("model"):

    row = {
        "model": model_name,
        "n_seeds": len(group)
    }

    for metric in metrics:
        values = pd.to_numeric(
            group[metric],
            errors="coerce"
        )

        row[f"{metric}_mean"] = values.mean()
        row[f"{metric}_sd"] = values.std(ddof=1)
        row[f"{metric}_min"] = values.min()
        row[f"{metric}_max"] = values.max()

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    SAVE_DIR / "transformer_multiseed_summary.csv",
    index=False
)

print("\n=== MULTI-SEED RUNS ===")
print(
    runs_df[
        ["model", "seed", "best_epoch",
         "accuracy", "f1", "errors", "fp", "fn"]
    ]
    .sort_values(["model", "seed"])
    .to_string(index=False)
)

print("\n=== MULTI-SEED SUMMARY ===")
print(summary_df.to_string(index=False))


RUNNING: CodeBERT | seed=7

corrected_group_split_multiseed | CodeBERT | seed=7 | train=45,288, val=5,679, test=5,654


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/4: loss=0.062731, val_f1=0.999103, val_acc=0.999296
Epoch 2/4: loss=0.004070, val_f1=0.999327, val_acc=0.999472
Epoch 3/4: loss=0.001036, val_f1=0.999327, val_acc=0.999472
Epoch 4/4: loss=0.000267, val_f1=0.999327, val_acc=0.999472
TEST: acc=0.999469, f1=0.999320, roc_auc=0.999997, pr_auc=0.999995, bal_acc=0.999483, mcc=0.998885, errors=3, FP=2, FN=1

SAVED AFTER THIS SEED:
/kaggle/working/sqlshield_corrected_validation/transformer_multiseed_runs.csv

RUNNING: CodeBERT | seed=21

corrected_group_split_multiseed | CodeBERT | seed=21 | train=45,288, val=5,679, test=5,654


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/4: loss=0.064350, val_f1=0.997756, val_acc=0.998239
Epoch 2/4: loss=0.004264, val_f1=0.998653, val_acc=0.998943
Epoch 3/4: loss=0.002098, val_f1=0.999102, val_acc=0.999296
Epoch 4/4: loss=0.000787, val_f1=0.999102, val_acc=0.999296
TEST: acc=0.999293, f1=0.999092, roc_auc=0.999649, pr_auc=0.999663, bal_acc=0.999093, mcc=0.998513, errors=4, FP=0, FN=4

SAVED AFTER THIS SEED:
/kaggle/working/sqlshield_corrected_validation/transformer_multiseed_runs.csv

RUNNING: BERT-base | seed=126

corrected_group_split_multiseed | BERT-base | seed=126 | train=45,288, val=5,679, test=5,654


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/4: loss=0.061811, val_f1=0.999102, val_acc=0.999296
Epoch 2/4: loss=0.003984, val_f1=0.998878, val_acc=0.999120


In [1]:
print("session started")

session started


In [2]:
from pathlib import Path
import pandas as pd

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")
RESULT_FILE = SAVE_DIR / "transformer_multiseed_runs.csv"

print("SAVE_DIR exists:", SAVE_DIR.exists())
print("Results file exists:", RESULT_FILE.exists())

if RESULT_FILE.exists():
    runs = pd.read_csv(RESULT_FILE)

    print("\n=== SAVED RUNS ===")
    print(
        runs[
            ["model", "seed", "best_epoch",
             "accuracy", "f1", "errors", "fp", "fn"]
        ]
        .sort_values(["model", "seed"])
        .to_string(index=False)
    )

    print("\n=== COMPLETED SEEDS ===")
    print(
        runs.groupby(["model", "seed"])
            .size()
            .to_string()
    )

SAVE_DIR exists: False
Results file exists: False


In [3]:
!kaggle kernels output mohammadalkhazaleh/notebook691d0eee69 -p /kaggle/working/version4_restore

Output file downloaded to /kaggle/working/version4_restore/.virtual_documents/__notebook_source__.ipynb
Output file downloaded to /kaggle/working/version4_restore/sqlshield_after_bert.zip
Output file downloaded to /kaggle/working/version4_restore/sqlshield_after_codebert.zip
Output file downloaded to /kaggle/working/version4_restore/sqlshield_corrected_validation/bert_seed42_result.json
Output file downloaded to /kaggle/working/version4_restore/sqlshield_corrected_validation/checkpoints/corrected_group_split_pilot__BERT-base__seed42.pt
Output file downloaded to /kaggle/working/version4_restore/sqlshield_corrected_validation/checkpoints/corrected_group_split_pilot__CodeBERT__seed42.pt
Output file downloaded to /kaggle/working/version4_restore/sqlshield_corrected_validation/codebert_seed42_result.json
Output file downloaded to /kaggle/working/version4_restore/sqlshield_corrected_validation/histories/corrected_group_split_pilot__BERT-base__seed42.csv
Output file downloaded to /kaggle/work

In [4]:
import shutil
from pathlib import Path

SRC = Path(
    "/kaggle/working/version4_restore/sqlshield_corrected_validation"
)

DST = Path(
    "/kaggle/working/sqlshield_corrected_validation"
)

if DST.exists():
    shutil.rmtree(DST)

shutil.copytree(SRC, DST)

print("Restored to:")
print(DST)

print("\nFiles:")
for f in sorted(DST.rglob("*")):
    if f.is_file():
        print(f)

Restored to:
/kaggle/working/sqlshield_corrected_validation

Files:
/kaggle/working/sqlshield_corrected_validation/bert_seed42_result.json
/kaggle/working/sqlshield_corrected_validation/checkpoints/corrected_group_split_pilot__BERT-base__seed42.pt
/kaggle/working/sqlshield_corrected_validation/checkpoints/corrected_group_split_pilot__CodeBERT__seed42.pt
/kaggle/working/sqlshield_corrected_validation/codebert_seed42_result.json
/kaggle/working/sqlshield_corrected_validation/histories/corrected_group_split_pilot__BERT-base__seed42.csv
/kaggle/working/sqlshield_corrected_validation/histories/corrected_group_split_pilot__CodeBERT__seed42.csv
/kaggle/working/sqlshield_corrected_validation/pilot_exact_mcnemar.csv
/kaggle/working/sqlshield_corrected_validation/predictions/classical__Random_Forest.csv
/kaggle/working/sqlshield_corrected_validation/predictions/corrected_group_split_pilot__BERT-base__seed42.csv
/kaggle/working/sqlshield_corrected_validation/predictions/corrected_group_split_pilo

In [5]:
from pathlib import Path
import pandas as pd
import json

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

train_df = pd.read_csv(SAVE_DIR / "train_corrected.csv")
val_df   = pd.read_csv(SAVE_DIR / "val_corrected.csv")
test_df  = pd.read_csv(SAVE_DIR / "test_corrected.csv")

with open(SAVE_DIR / "codebert_seed42_result.json") as f:
    cb42 = json.load(f)

with open(SAVE_DIR / "bert_seed42_result.json") as f:
    b42 = json.load(f)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("CodeBERT seed42 errors:", cb42["errors"])
print("CodeBERT seed42 F1:", cb42["f1"])

print("BERT seed42 errors:", b42["errors"])
print("BERT seed42 F1:", b42["f1"])

print(
    "train_transformer_once available:",
    "train_transformer_once" in globals()
)

Train: 45288
Validation: 5679
Test: 5654
CodeBERT seed42 errors: 3
CodeBERT seed42 F1: 0.9993192648059904
BERT seed42 errors: 6
BERT seed42 F1: 0.9986382206082615
train_transformer_once available: False


In [8]:
print(
    "train_transformer_once available:",
    "train_transformer_once" in globals()
)

print("DEVICE:", DEVICE)

train_transformer_once available: True
DEVICE: cuda


In [9]:
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

expected = []

for model in ["CodeBERT", "BERT-base"]:
    for seed in [7, 21, 42, 84, 126]:

        model_file = model.replace("-", "-")

        if seed == 42:
            pred = SAVE_DIR / "predictions" / \
                f"corrected_group_split_pilot__{model_file}__seed42.csv"
            hist = SAVE_DIR / "histories" / \
                f"corrected_group_split_pilot__{model_file}__seed42.csv"
            ckpt = SAVE_DIR / "checkpoints" / \
                f"corrected_group_split_pilot__{model_file}__seed42.pt"
        else:
            pred = SAVE_DIR / "predictions" / \
                f"corrected_group_split_multiseed__{model_file}__seed{seed}.csv"
            hist = SAVE_DIR / "histories" / \
                f"corrected_group_split_multiseed__{model_file}__seed{seed}.csv"
            ckpt = SAVE_DIR / "checkpoints" / \
                f"corrected_group_split_multiseed__{model_file}__seed{seed}.pt"

        expected.append({
            "model": model,
            "seed": seed,
            "prediction": pred.exists(),
            "history": hist.exists(),
            "checkpoint": ckpt.exists()
        })

import pandas as pd

audit = pd.DataFrame(expected)

print("\n=== MULTI-SEED FILE AUDIT ===")
print(audit.to_string(index=False))


=== MULTI-SEED FILE AUDIT ===
    model  seed  prediction  history  checkpoint
 CodeBERT     7       False    False       False
 CodeBERT    21       False    False       False
 CodeBERT    42        True     True        True
 CodeBERT    84       False    False       False
 CodeBERT   126       False    False       False
BERT-base     7       False    False       False
BERT-base    21       False    False       False
BERT-base    42        True     True        True
BERT-base    84       False    False       False
BERT-base   126       False    False       False


In [ ]:
import json
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

codebert_7 = train_transformer_once(
    experiment="corrected_group_split_multiseed",
    model_name="CodeBERT",
    model_id="microsoft/codebert-base",
    seed=7,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
)

# Save this seed independently
with open(
    SAVE_DIR / "codebert_seed7_result.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        codebert_7,
        f,
        indent=2,
        default=str
    )

print("\n=== CODEBERT SEED 7 ===")
for k, v in codebert_7.items():
    print(f"{k}: {v}")

print("\nSaved independently:")
print(SAVE_DIR / "codebert_seed7_result.json")


corrected_group_split_multiseed | CodeBERT | seed=7 | train=45,288, val=5,679, test=5,654


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]